In [1]:
import pandas as pd
import numpy as np

In [2]:
# ============================================================
# PHASE 1: Data Loading, Validation & Feature Engineering
# Afficionado Coffee Roasters - Sales Trend Analysis
# ============================================================

In [3]:
# ============================================================
# STEP 1: LOAD DATA
# ============================================================
# Place your downloaded CSV in the same folder as this script
# and rename it to coffee_sales.csv
 
df = pd.read_csv("data/coffee_sales.csv")
 
print("=" * 50)
print("STEP 1: RAW DATA OVERVIEW")
print("=" * 50)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst 3 rows:\n{df.head(3)}")

STEP 1: RAW DATA OVERVIEW
Shape: 149116 rows x 11 columns

Column names:
['transaction_id', 'year', 'transaction_time', 'transaction_qty', 'store_id', 'store_location', 'product_id', 'unit_price', 'product_category', 'product_type', 'product_detail']

Data types:
transaction_id        int64
year                  int64
transaction_time        str
transaction_qty       int64
store_id              int64
store_location          str
product_id            int64
unit_price          float64
product_category        str
product_type            str
product_detail          str
dtype: object

First 3 rows:
   transaction_id  year transaction_time  transaction_qty  store_id  \
0               1  2025          7:06:11                2         5   
1               2  2025          7:08:56                2         5   
2               3  2025          7:14:04                2         5   

    store_location  product_id  unit_price    product_category  \
0  Lower Manhattan          32         3.0      

In [4]:
# ============================================================
# STEP 2: VALIDATION CHECKS
# ============================================================
 
print("\n" + "=" * 50)
print("STEP 2: VALIDATION CHECKS")
print("=" * 50)
 
# --- 2a. Missing values ---
missing = df.isnull().sum()
print(f"\nMissing values per column:\n{missing}")
assert missing.sum() == 0, "WARNING: Missing values found! Check above."
print("✓ No missing values.")
 
# --- 2b. Duplicate transaction IDs ---
dup_count = df["transaction_id"].duplicated().sum()
print(f"\nDuplicate transaction_ids: {dup_count}")
assert dup_count == 0, "WARNING: Duplicate transaction_ids found!"
print("✓ No duplicate transaction IDs.")
 
# --- 2c. Sequential check (should run 1 to 149456) ---
expected_ids = set(range(1, df["transaction_id"].max() + 1))
actual_ids   = set(df["transaction_id"])
missing_ids  = expected_ids - actual_ids
print(f"\nExpected ID range: 1 to {df['transaction_id'].max()}")
print(f"Missing IDs in sequence: {len(missing_ids)}")
if missing_ids:
    print(f"  Sample missing IDs: {sorted(missing_ids)[:10]}")
 
# --- 2d. Logical consistency: positive qty and price ---
neg_qty   = (df["transaction_qty"] <= 0).sum()
neg_price = (df["unit_price"] <= 0).sum()
print(f"\nRows with transaction_qty <= 0 : {neg_qty}")
print(f"Rows with unit_price <= 0     : {neg_price}")
assert neg_qty   == 0, "WARNING: Non-positive quantities found!"
assert neg_price == 0, "WARNING: Non-positive prices found!"
print("✓ All quantities and prices are positive.")
 
# --- 2e. year column should be all 2025 ---
unique_years = df["year"].unique()
print(f"\nUnique years in dataset: {unique_years}")
assert list(unique_years) == [2025], "WARNING: Unexpected year values found!"
print("✓ All transactions are from 2025.")
 
# --- 2f. transaction_time format check ---
sample_times = df["transaction_time"].head(5).tolist()
print(f"\nSample transaction_time values: {sample_times}")


STEP 2: VALIDATION CHECKS

Missing values per column:
transaction_id      0
year                0
transaction_time    0
transaction_qty     0
store_id            0
store_location      0
product_id          0
unit_price          0
product_category    0
product_type        0
product_detail      0
dtype: int64
✓ No missing values.

Duplicate transaction_ids: 0
✓ No duplicate transaction IDs.

Expected ID range: 1 to 149456
Missing IDs in sequence: 340
  Sample missing IDs: [3252, 3253, 3254, 3255, 3256, 3257, 3258, 3259, 3260, 3261]

Rows with transaction_qty <= 0 : 0
Rows with unit_price <= 0     : 0
✓ All quantities and prices are positive.

Unique years in dataset: [2025]
✓ All transactions are from 2025.

Sample transaction_time values: ['7:06:11', '7:08:56', '7:14:04', '7:20:24', '7:22:41']


In [5]:
# ============================================================
# STEP 3: PARSE transaction_time
# ============================================================
 
print("\n" + "=" * 50)
print("STEP 3: PARSING TIMESTAMPS")
print("=" * 50)
 
# Handles both '7:06:11' and '07:06:11' formats safely
df["transaction_time_parsed"] = pd.to_datetime(
    df["transaction_time"], format="mixed"
)
 
print(f"Parsed time sample:\n{df['transaction_time_parsed'].head(5)}")
print(f"\nMin time: {df['transaction_time_parsed'].dt.time.min()}")
print(f"Max time: {df['transaction_time_parsed'].dt.time.max()}")
 
# Verify no NaT (failed parses)
nat_count = df["transaction_time_parsed"].isna().sum()
print(f"Failed parses (NaT): {nat_count}")
assert nat_count == 0, "WARNING: Some transaction_time values failed to parse!"
print("✓ All timestamps parsed successfully.")


STEP 3: PARSING TIMESTAMPS
Parsed time sample:
0   2026-06-07 07:06:11
1   2026-06-07 07:08:56
2   2026-06-07 07:14:04
3   2026-06-07 07:20:24
4   2026-06-07 07:22:41
Name: transaction_time_parsed, dtype: datetime64[us]

Min time: 06:00:00
Max time: 20:59:32
Failed parses (NaT): 0
✓ All timestamps parsed successfully.


In [6]:
# ============================================================
# STEP 4: FEATURE ENGINEERING
# ============================================================
 
print("\n" + "=" * 50)
print("STEP 4: FEATURE ENGINEERING")
print("=" * 50)
 
# --- 4a. Revenue per transaction ---
df["revenue"] = df["transaction_qty"] * df["unit_price"]
print(f"\nRevenue column created.")
print(f"  Min: ${df['revenue'].min():.2f}")
print(f"  Max: ${df['revenue'].max():.2f}")
print(f"  Total: ${df['revenue'].sum():,.2f}")
 
# --- 4b. Hour of day (0–23) ---
df["hour"] = df["transaction_time_parsed"].dt.hour
print(f"\nHour range: {df['hour'].min()} to {df['hour'].max()}")
 
# --- 4c. Day of week (0=Monday ... 6=Sunday) as number and label ---
# Using transaction_id as sequential proxy for day ordering
# We bin 149,456 transactions across 365 days (2025)
# Each "day bin" = ~409 transactions
TOTAL_ROWS = len(df)
DAYS_IN_YEAR = 365
 
df = df.sort_values("transaction_id").reset_index(drop=True)
df["day_bin"] = (df.index // (TOTAL_ROWS / DAYS_IN_YEAR)).astype(int)
df["day_bin"] = df["day_bin"].clip(0, DAYS_IN_YEAR - 1)  # safety clip
 
# 2025 starts on Wednesday (weekday=2)
# day_of_week_num: 0=Monday, 6=Sunday
df["day_of_week_num"]  = (df["day_bin"] + 2) % 7    # +2 because Jan 1 2025 = Wednesday
DAY_NAMES = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["day_of_week"] = df["day_of_week_num"].map(lambda x: DAY_NAMES[x])
 
print(f"\nDay of week distribution:")
print(df["day_of_week"].value_counts().reindex(DAY_NAMES))
 
# --- 4d. Week number (1–52) for trend analysis ---
df["week_number"] = (df["day_bin"] // 7) + 1
df["week_number"] = df["week_number"].clip(1, 52)
print(f"\nWeek number range: {df['week_number'].min()} to {df['week_number'].max()}")
 
# --- 4e. Time bucket ---
def assign_time_bucket(hour):
    if   6  <= hour <= 11: return "Morning (6–11)"
    elif 12 <= hour <= 16: return "Afternoon (12–16)"
    elif 17 <= hour <= 21: return "Evening (17–21)"
    else:                  return "Late Hours (22–5)"
 
df["time_bucket"] = df["hour"].apply(assign_time_bucket)
print(f"\nTime bucket distribution:")
print(df["time_bucket"].value_counts())
 
# --- 4f. Is Weekend flag ---
df["is_weekend"] = df["day_of_week"].isin(["Saturday", "Sunday"])
print(f"\nWeekend transactions : {df['is_weekend'].sum():,}")
print(f"Weekday transactions : {(~df['is_weekend']).sum():,}")


STEP 4: FEATURE ENGINEERING

Revenue column created.
  Min: $0.80
  Max: $360.00
  Total: $698,812.33

Hour range: 6 to 20

Day of week distribution:
day_of_week
Monday       21244
Tuesday      21244
Wednesday    21652
Thursday     21244
Friday       21244
Saturday     21244
Sunday       21244
Name: count, dtype: int64

Week number range: 1 to 52

Time bucket distribution:
time_bucket
Morning (6–11)       81751
Afternoon (12–16)    44427
Evening (17–21)      22938
Name: count, dtype: int64

Weekend transactions : 42,488
Weekday transactions : 106,628


In [7]:
# ============================================================
# STEP 5: FINAL CLEAN DATAFRAME SUMMARY
# ============================================================
 
print("\n" + "=" * 50)
print("STEP 5: FINAL DATAFRAME SUMMARY")
print("=" * 50)
 
# Drop the intermediate parsed time column (keep only what's needed)
df_clean = df.drop(columns=["transaction_time_parsed"])
 
print(f"\nFinal shape   : {df_clean.shape}")
print(f"\nAll columns   : {df_clean.columns.tolist()}")
print(f"\nSample row:\n{df_clean.iloc[0]}")
print(f"\nData types:\n{df_clean.dtypes}")
 
# Quick sanity stats
print("\n--- Revenue Summary ---")
print(f"Total revenue     : ${df_clean['revenue'].sum():>12,.2f}")
print(f"Avg per txn       : ${df_clean['revenue'].mean():>12.2f}")
print(f"Total transactions: {len(df_clean):>12,}")
 
print("\n--- Store Locations ---")
print(df_clean["store_location"].value_counts())
 
print("\n--- Product Categories ---")
print(df_clean["product_category"].value_counts())


STEP 5: FINAL DATAFRAME SUMMARY

Final shape   : (149116, 19)

All columns   : ['transaction_id', 'year', 'transaction_time', 'transaction_qty', 'store_id', 'store_location', 'product_id', 'unit_price', 'product_category', 'product_type', 'product_detail', 'revenue', 'hour', 'day_bin', 'day_of_week_num', 'day_of_week', 'week_number', 'time_bucket', 'is_weekend']

Sample row:
transaction_id                          1
year                                 2025
transaction_time                  7:06:11
transaction_qty                         2
store_id                                5
store_location            Lower Manhattan
product_id                             32
unit_price                            3.0
product_category                   Coffee
product_type        Gourmet brewed coffee
product_detail                Ethiopia Rg
revenue                               6.0
hour                                    7
day_bin                                 0
day_of_week_num                  

In [8]:
# ============================================================
# STEP 6: SAVE CLEAN DATA
# ============================================================
 
df_clean.to_csv("data/coffee_sales_clean.csv", index=False)
print("\n" + "=" * 50)
print("✓ PHASE 1 COMPLETE")
print("  Saved: data/coffee_sales_clean.csv")
print("  Rows  :", len(df_clean))
print("  Cols  :", len(df_clean.columns))
print("=" * 50)


✓ PHASE 1 COMPLETE
  Saved: data/coffee_sales_clean.csv
  Rows  : 149116
  Cols  : 19
